In [ ]:
# ========== ИМПОРТЫ ==========

import re
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from tqdm import tqdm
import gc
import warnings
import os
import time
import yaml
import re
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader

# NLP библиотеки
import nltk
from nltk.tokenize import word_tokenize
import emoji

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# Скачать токенизатор (один раз)
nltk.download('punkt')

warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split

# ROUGE метрика
from rouge_score import rouge_scorer

# Transformers для сравнения
from transformers import pipeline

In [ ]:
# ========== 1. КОНФИГУРАЦИЯ ==========
class Config:
    # Параметры данных
    min_word_freq = 2
    
    # Параметры модели LSTM
    lstm_embedding_dim = 128
    lstm_hidden_dim = 256
    lstm_num_layers = 2
    lstm_dropout = 0.3
    lstm_batch_size = 128
    lstm_epochs = 3
    lstm_lr = 0.001
    lstm_weight_decay = 1e-5
    
    # Параметры генерации
    max_seq_len = 80
    gen_temperature = 0.8
    gen_top_k = 50
    gen_top_p = 0.9
    
    # Другое
    random_seed = 42
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = Config()
print("="*60)
print("ФИНАЛЬНЫЙ СКРИПТ: LSTM vs DistilGPT2 (KAGGLE)")
print("="*60)
print(f"Используется устройство: {config.device}")
if config.device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Память GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
random.seed(config.random_seed)
np.random.seed(config.random_seed)
torch.manual_seed(config.random_seed)

In [ ]:
# ========== 2. ЗАГРУЗКА И ОЧИСТКА ДАННЫХ ==========
print("\n" + "="*60)
print("ЭТАП 1: ЗАГРУЗКА И ОЧИСТКА ДАННЫХ")
print("="*60)

# Скачиваем nltk данные
try:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
except:
    pass

# Загружаем файл
file_path = '/kaggle/input/datasets/viacheslavr/raw-dataset-txt/raw_dataset.txt'

with open(file_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

# Убираем пустые строки
raw_texts = [line.strip() for line in lines if line.strip()]
print(f"Загружено текстов: {len(raw_texts)}")
print("Пример:", raw_texts[0])

def clean_text(text):
    # Удалить упоминания
    text = re.sub(r'@\w+', '', text)
    # Удалить ссылки
    text = re.sub(r'http\S+', '', text)
    # Удалить эмодзи
    text = emoji.replace_emoji(text, replace='')
    # Привести к нижнему регистру
    text = text.lower()
    # Удалить лишние пробелы
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Очищаем
print("Очистка текстов...")
cleaned_texts = [clean_text(text) for text in tqdm(raw_texts)]

print("Оригинал:", raw_texts[0])
print("Очищенный:", cleaned_texts[0])

# Токенизация
print("\nТокенизация...")
tokenized_texts = [word_tokenize(text) for text in tqdm(cleaned_texts)]
print("Токены:", tokenized_texts[0][:10])

# Удаляем пустые тексты
non_empty = [i for i, tokens in enumerate(tokenized_texts) if len(tokens) > 0]
cleaned_texts = [cleaned_texts[i] for i in non_empty]
tokenized_texts = [tokenized_texts[i] for i in non_empty]
print(f"После фильтрации: {len(cleaned_texts)} текстов")

In [ ]:
# ========== 3. РАЗДЕЛЕНИЕ НА ТРЕЙН/ВАЛ/ТЕСТ (80/10/10) ==========
print("\n" + "="*60)
print("ЭТАП 1 (продолжение): РАЗДЕЛЕНИЕ ДАННЫХ")
print("="*60)

# Сначала разделяем на трейн (80%) и временные (20%)
train_texts, temp_texts, train_tokens, temp_tokens = train_test_split(
    cleaned_texts, tokenized_texts, 
    test_size=0.2, 
    random_state=config.random_seed
)

# Затем временные на вал (10%) и тест (10%)
val_texts, test_texts, val_tokens, test_tokens = train_test_split(
    temp_texts, temp_tokens, 
    test_size=0.5, 
    random_state=config.random_seed
)

print(f"Train: {len(train_texts)} ({len(train_texts)/len(cleaned_texts)*100:.1f}%)")
print(f"Validation: {len(val_texts)} ({len(val_texts)/len(cleaned_texts)*100:.1f}%)")
print(f"Test: {len(test_texts)} ({len(test_texts)/len(cleaned_texts)*100:.1f}%)")

In [ ]:
# ========== 4. ПОСТРОЕНИЕ СЛОВАРЯ ==========
print("\n" + "="*60)
print("ЭТАП 1 (продолжение): ПОСТРОЕНИЕ СЛОВАРЯ")
print("="*60)

def build_vocab(tokens_list, min_freq=2):
    word_counts = Counter()
    for tokens in tokens_list:
        word_counts.update(tokens)
    
    vocab = {'<PAD>': 0, '<UNK>': 1}
    idx = 2
    for word, count in word_counts.items():
        if count >= min_freq:
            vocab[word] = idx
            idx += 1
    
    return vocab, word_counts

vocab, word_counts = build_vocab(train_tokens, min_freq=config.min_word_freq)
print(f"Размер словаря: {len(vocab)}")
print(f"Токенов в обучающей выборке: {sum(word_counts.values()):,}")

# Обратный словарь
idx_to_word = {v: k for k, v in vocab.items()}

In [ ]:
# ========== 5. ПОДГОТОВКА DATASET И DATALOADER ==========
print("\n" + "="*60)
print("ЭТАП 1 (продолжение): ПОДГОТОВКА DATALOADER")
print("="*60)

def tokens_to_indices(tokens, vocab):
    return [vocab.get(token, vocab['<UNK>']) for token in tokens]

class TextDataset(Dataset):
    def __init__(self, texts, tokens, vocab):
        self.texts = texts
        self.indices = [torch.tensor(tokens_to_indices(t, vocab), dtype=torch.long) 
                       for t in tokens]
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        return {
            'text': self.texts[idx],
            'indices': self.indices[idx]
        }

def collate_batch(batch):
    texts = [item['text'] for item in batch]
    indices = [item['indices'] for item in batch]
    
    # Обрезаем длинные последовательности
    indices = [seq[:config.max_seq_len] for seq in indices]
    indices_padded = pad_sequence(indices, batch_first=True, padding_value=0)
    
    return {
        'texts': texts,
        'indices': indices_padded
    }

# Создаем датасеты
train_dataset = TextDataset(train_texts, train_tokens, vocab)
val_dataset = TextDataset(val_texts, val_tokens, vocab)
test_dataset = TextDataset(test_texts, test_tokens, vocab)

# Создаем DataLoader'ы
train_loader = DataLoader(
    train_dataset, 
    batch_size=config.lstm_batch_size, 
    shuffle=True, 
    collate_fn=collate_batch,
    pin_memory=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=config.lstm_batch_size, 
    shuffle=False, 
    collate_fn=collate_batch,
    pin_memory=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=config.lstm_batch_size, 
    shuffle=False, 
    collate_fn=collate_batch,
    pin_memory=True,
    num_workers=2
)

print(f"Train батчей: {len(train_loader)}")
print(f"Val батчей: {len(val_loader)}")
print(f"Test батчей: {len(test_loader)}")